In [1]:
import os
import sys
import numpy as np
from PIL import Image
from torchvision.transforms import ToTensor
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import cv2
from PIL import Image

In [2]:
class OCRNet(nn.Module):
    def __init__(self, num_classes: int):
        super(OCRNet, self).__init__()
        self.features = nn.Sequential(
            # 32,768 neuroni output = 32 de filtre=> 32 new pixal maps=> 32x32 neuroni(1:1 valoare pixel)
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # 1: grayscale, 32: outputfilters, 3x3 kernels
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),          

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),          
            #2048 neuroni
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),         
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), # Vector 1D de 2048( 128 pixel maps [4x4]  )
            nn.Linear(2048, 256), #invata curbele si cui caracter corespund// fortarea compact a dimensiunii si valorilor regulate
            nn.ReLU(inplace=True), #non-liniaritatea
            nn.Dropout(0.5), #necesar pt prevenirea invatarii zgomotului si imaginilor la exact
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [34]:
def preprocess_word_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Imagine lipsa: {path}")
    _, thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU) # conversie grayscale la o masca binara / inversare culoare litere
    #pixelii mai mici ca 0+cv.THRESH_OTSU devin 0(negri), pixelii mai mari ca 0+cv.THRESH_OTSU devini 255(albi)
    #cv2.THRESH_BINARY_INV - scris alb/ background negru
    #cv2.THRESH_OTSU- analizeaza imaginea si modifica dinamic marginea(0) alesa 

    return img, thresh
    #img= un array de h,l cu valori intre 0-255, segmentarea corecta a literelor

In [35]:
def segment_characters(thresh):
    # Find connected components (each character = one blob)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(thresh)
    #num_labels - nr total de segmente gasite
    #matrice de identificare
    #stats- coordonate, dimensiune per segment / stats[i] = [x, y, w, h, area], [x,y]-coltul stang , w-latime,h-inaltime
    boxes = []
    for i in range(1, num_labels): 
        x, y, w, h, area = stats[i]
        boxes.append((x, y, w, h))
    boxes = sorted(boxes, key=lambda b: b[0]) #sortarea segmentelor dupa pozitie folosind coordonatele x,y
    return boxes

In [36]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [37]:
checkpoint = torch.load("../PROIECT_2/checkpoints/best_ocr_model.pth", map_location="cpu", weights_only=False)
class_names = checkpoint["class_names"]
model = OCRNet(num_classes=len(class_names))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model incarcat")

Model incarcat


In [38]:
def predict_character(img):
    x = transform(img).unsqueeze(0)
    with torch.no_grad():
        outputs = model(x)
        _, predicted = outputs.max(1)
    return class_names[predicted.item()]

In [39]:
def predict_word(path):
    original_img, thresh = preprocess_word_image(path)
    boxes = segment_characters(thresh)

    word = ""
    for (x, y, w, h) in boxes:
        char_img = original_img[y:y+h, x:x+w]          
        char_img = Image.fromarray(char_img).convert("L")     
        char_img = char_img.resize((32, 32))
        char = predict_character(char_img)
        word += char

    return word

In [40]:
img = Image.open("testing_data/N/28837.png").convert("L")
print("Predicted:", predict_character(img))

Predicted: N


In [41]:
print("Word: ", predict_word("tst4.png"))

Word:  MISASTRICATMASINA


In [46]:
def predict_sentence(path, gap_threshold=12):
    original_img, thresh = preprocess_word_image(path)
    boxes = segment_characters(thresh)
    words = []
    current_word = []
    for i in range(len(boxes)):
        current_word.append(boxes[i])
        #comparare caracter actual cu urmator
        if i < len(boxes) - 1:
            x, y, w, h = boxes[i]
            nx, ny, nw, nh = boxes[i + 1]
            gap = nx - (x + w)
            if gap > gap_threshold:
                words.append(current_word)
                current_word = []
    if current_word:
        words.append(current_word)
    sentence = []
    for word_boxes in words:
        predicted_word = ""
        for (x, y, w, h) in word_boxes:
            char_img = original_img[y:y+h, x:x+w]
            char_img = Image.fromarray(char_img).convert("L")
            predicted_word += predict_character(char_img)

        sentence.append(predicted_word)
    return " ".join(sentence)


In [47]:
print("Word: ", predict_sentence("tst1.png"))

Word:  CIOCOLATA ESTE FOARTE BUN4
